# chb01 data preparation with ChatGPT

 전체 구성 요약
1. 데이터 로딩
`mne.io.read_raw_edf()`로 `chb01_*.edf` 파일 읽기

텍스트 파일에서 발작 구간 정보 파싱

2. 발작 구간 자동 주석화
텍스트 파일에서 `seizure start/end`를 읽고 MNE `Annotations` 객체로 변환

3. 고정 타임 윈도우(10초, stride 2초)로 슬라이딩 윈도우 나누기
각 window를 `ictal` or `interictal`로 레이블링

In [7]:
import os
import mne
import numpy as np
from pathlib import Path

# ---------- 설정 ----------
subject_id = "chb01"
edf_dir = Path("/home/media/data2/user_home/khj/KHJworks/EEGstudy/EESNN/physionet.org/files/chbmit/1.0.0/chb01")  # .edf 파일 위치
summary_file = Path("/home/media/data2/user_home/khj/KHJworks/EEGstudy/EESNN/physionet.org/files/chbmit/1.0.0/chb01/chb01-summary.txt")  # seizure summary 텍스트파일
window_size_sec = 10
stride_sec = 2
sfreq = 256  # Hz

In [ ]:
# ---------- 1. 발작 정보 읽기 ----------

def parse_seizure_summary(summary_path):
    seizure_dict = {}
    current_file = None

    with open(summary_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("File Name:"):
                current_file = line.split(":")[1].strip()
                seizure_dict[current_file] = []
            elif line.startswith("Number of Seizures in File:"):
                continue
            elif line.startswith("Seizure Start Time:"):
                time_str = line.split(":")[1].strip().replace(" seconds", "")
                start_time = float(time_str)
            elif line.startswith("Seizure End Time:"):
                time_str = line.split(":")[1].strip().replace(" seconds", "")
                end_time = float(time_str)
                seizure_dict[current_file].append((start_time, end_time))
    return seizure_dict

seizure_info = parse_seizure_summary(summary_file)


In [9]:
# ---------- 2. 파일별로 MNE 로딩 + 주석 처리 ----------
def load_raw_with_annotations(edf_path, seizure_times):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    if seizure_times:
        annotations = mne.Annotations(
            onset=[start for (start, end) in seizure_times],
            duration=[end - start for (start, end) in seizure_times],
            description=['seizure'] * len(seizure_times)
        )
        raw.set_annotations(annotations)
    return raw

In [10]:
# ---------- 3. 슬라이딩 윈도우 생성 + 레이블링 ----------
def sliding_windows(raw, window_sec, stride_sec):
    window_samples = int(window_sec * raw.info['sfreq'])
    stride_samples = int(stride_sec * raw.info['sfreq'])
    total_samples = raw.n_times
    labels = []
    segments = []

    # 주석 가져오기
    seizure_onsets = []
    seizure_ends = []
    if raw.annotations:
        for ann in raw.annotations:
            if ann['description'] == 'seizure':
                seizure_onsets.append(ann['onset'] * raw.info['sfreq'])
                seizure_ends.append((ann['onset'] + ann['duration']) * raw.info['sfreq'])

    for start in range(0, total_samples - window_samples + 1, stride_samples):
        stop = start + window_samples
        segment = raw.get_data(start=start, stop=stop)

        # ictal 여부 판단
        is_ictal = False
        for onset, end in zip(seizure_onsets, seizure_ends):
            if start < end and stop > onset:  # 윈도우가 발작 구간과 겹치면 ictal
                is_ictal = True
                break
        label = 'ictal' if is_ictal else 'nonictal'
        segments.append(segment)
        labels.append(label)

    return np.array(segments), np.array(labels)

In [ ]:
# ---------- 4. 전체 데이터 처리 루프 ----------

all_segments = []
all_labels = []

for edf_file, seizure_times in seizure_info.items():
    edf_path = edf_dir / edf_file
    if not edf_path.exists():
        continue
    print(f"Processing {edf_file}...")
    raw = load_raw_with_annotations(edf_path, seizure_times)
    segments, labels = sliding_windows(raw, window_size_sec, stride_sec)
    all_segments.append(segments)
    all_labels.append(labels)

# 전체 chb01 파일 처리 및 최종 데이터셋 X(EEG 시퀀스), y(레이블) 생성
X = np.concatenate(all_segments, axis=0)
y = np.concatenate(all_labels, axis=0)
print(f"전체 윈도우 수: {len(y)} / ictal: {(y == 'ictal').sum()} / nonictal: {(y == 'nonictal').sum()}")



Processing chb01_01.edf...


/tmp/ipykernel_756155/1873271709.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Processing chb01_02.edf...


/tmp/ipykernel_756155/1873271709.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Processing chb01_03.edf...


...

Processing chb01_43.edf...


/tmp/ipykernel_756155/1873271709.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Processing chb01_46.edf...


/tmp/ipykernel_756155/1873271709.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


전체 윈도우 수: 72825 / ictal: 251 / nonictal: 72574


※
1. 전체 윈도우 수: 72825 / ictal: 251 / nonictal: 72574
척 봐도 데이터 불균형 수준 폼 미침
이제 이거를 어떻게 해결할까? - 언더샘플링을 해보자
>>> !! 이 정도 비율이면 1:290 정도로 불균형이 매우 심한 편이야.!! 이런 경우, 모델은 무조건 'nonictal'이라고만 예측해도 정확도가 높게 나오는 착시를 일으킬 수 있어. 실제 중요한 발작 구간은 거의 감지하지 못하게 될 가능성이 커.

※ 불균형 데이터 처리법; 언더샘플링 설명 자료
https://bommbom.tistory.com/entry/%EB%B6%88%EA%B7%A0%ED%98%95-%EB%8D%B0%EC%9D%B4%ED%84%B0Data-Imbalance-%EC%B2%98%EB%A6%AC-%EC%96%B8%EB%8D%94-%EC%83%98%ED%94%8C%EB%A7%81Under-Sampling

[전략]
- 언더샘플링(비발작 데이터 줄이기) 
  무던하게 진행할 수 았는 방법
- 오버샘플링(발작 데이터 늘리기) 
  얘는 영 좋지 않은 전략인 듯 
- 가중치 부여(손실 함수에 class weight; rare class(ictal?)에 더 큰 가중치 부여)
  양측 데이터 유지, 파이토치&브디코 모두 지원 >>> 각이다


2. 거슬리는 출력 문구 물어보기
RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False) << 이 경고가 무엇인지
  >>> 당장은 무시 가능.
  `mne.io.read_raw_edf()`는 EEG 채널 이름을 기준으로 데이터를 로딩하는데, 특정 EDF 파일에 동일한 이름의 채널이 두 번 이상 존재하면 충돌이 생길 수 있음. 이 경우 "T8-P8"라는 채널 이름이 두 번 등장했기 때문에, MNE는 자동으로 뒤에 오는 채널 이름에 번호를 붙여서 중복을 회피함.
  T8-P8 -> T8-P8-0
  T8-P8 -> T8-P8-1
  
  그러나, Braindecode 모델에 입력될 때 채널 순서나 이름을 기준으로 정렬하거나 선택할 계획이 있다면 적절한 수정 필요. (이미 -0, -1처럼 자동 수정되어서 ㄱㅊ할 듯?)


3. 1,2 끝나면 발작 정보 csv 포맷으로 출력 (0613 13:59 이제 이거 해야 되)


# 데이터 불균형 밸패; 전략 3. 가중치 부여
딥러닝 기반 브디코 모델 사용시 이 전략을 쓰는 것이 좋음 (필수 1단계 -> optional 2단계)

In [12]:
# ✅ 1단계: 데이터는 그대로 유지하고, loss function에 클래스 가중치(class weights) 적용

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 예시: 'ictal' = 1, 'nonictal' = 0
labels_numeric = np.where(y == 'ictal', 1, 0)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels_numeric), y=labels_numeric)
print("Class Weights:", class_weights)


Class Weights: [  0.50172927 145.06972112]


In [13]:
# ✅ 2단계: 필요시 언더샘플링 (참고용)

from sklearn.utils import resample

ictal_indices = np.where(y == 'ictal')[0]
nonictal_indices = np.where(y == 'nonictal')[0]

# nonictal에서 ictal과 같은 수만큼 샘플링
nonictal_down = resample(nonictal_indices, n_samples=len(ictal_indices), replace=False, random_state=42)

selected_indices = np.concatenate([ictal_indices, nonictal_down])
X_balanced = X[selected_indices]
y_balanced = y[selected_indices]

# 지피티: 이렇게 하면 클래스 균형을 맞출 수 있지만, 
# 데이터 손실 위험이 있으므로 오직 실험 목적이나 매우 빠른 프로토타이핑에만 추천해.

# 일단 연습용이니까 해봐야지 ㅎㅎ

gpt: 필요하다면 다음 단계로 Braindecode의 Deep4Net 모델에 이 balanced 데이터셋을 어떻게 넣는지도 도와줄게!
ㄴ 잠깐 타임
    seizure 정보 .csv 포맷이나 브디코 BaseConcatDataset 형식으로 변환하는 거 먼저 하고 옴 

✅ 1. seizure_dict를 .csv로 저장하기

In [ ]:

import pandas as pd

def save_seizure_info_to_csv(seizure_dict, save_path='/EESNN/eiesdief/chb01_seizure_info.csv'):
    rows = []
    for filename, seizures in seizure_dict.items():
        for seizure in seizures:
            start, end = seizure
            rows.append({
                'filename': filename,
                'seizure_start': start,
                'seizure_end': end
            })
    df = pd.DataFrame(rows)
    df.to_csv(save_path, index=False)
    print(f"Saved seizure info to {save_path}")
    

✅ 2. Braindecode용 BaseConcatDataset 형식으로 변환

`Braindecode`에서는 `WindowsDataset` 또는 `BaseConcatDataset`으로 데이터 관리를 해. 이걸 위해선 `Description`과 함께 `Raw` 객체 또는 `Epochs` 객체가 필요하고, 각 윈도우에 ictal/nonictal 라벨을 매겨야 해.

🔧 구조 설계
항목	                설명
raw	                    MNE의 Raw 객체 (edf 파일)
description	            ictal 여부 (0/1), 시작시간, 종료시간 등 메타정보
windows_dataset	        윈도우 단위 EEG 세그먼트 (10초, stride=2초)

In [ ]:
# 🧠 예시 코드: Raw + 라벨링 + 윈도우 생성 + BaseConcatDataset

from braindecode.datasets.base import BaseConcatDataset
from braindecode.preprocessing import create_fixed_length_windows
from mne import Annotations

def create_labeled_dataset(edf_paths, seizure_dict, window_size_sec=10, stride_sec=2, sfreq=256):
    datasets = []

    for edf_path in edf_paths:
        file_name = edf_path.split('/')[-1]
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

        # 라벨링용 Annotations 추가
        annotations = []
        for seizure_start, seizure_end in seizure_dict.get(file_name, []):
            annotations.append(dict(onset=seizure_start,
                                    duration=seizure_end - seizure_start,
                                    description='seizure'))
        if annotations:
            annots = mne.Annotations(onset=[a['onset'] for a in annotations],
                                     duration=[a['duration'] for a in annotations],
                                     description=[a['description'] for a in annotations])
            raw.set_annotations(annots)

        # Fixed-length windows 생성
        windows = create_fixed_length_windows(
            raw,
            start_offset_samples=0,
            stop_offset_samples=0,
            window_size_samples=int(window_size_sec * sfreq),
            stride_samples=int(stride_sec * sfreq),
            drop_last_window=False,
            preload=True
        )

        # 라벨링: 윈도우가 seizure annotation에 걸치면 ictal (1), 아니면 nonictal (0)
        y = []
        for desc in windows.description['annotated']:
            if isinstance(desc, str) and 'seizure' in desc:
                y.append(1)
            else:
                y.append(0)

        # description 추가
        windows.set_description(dict(y=y))
        datasets.append(windows)

    full_dataset = BaseConcatDataset(datasets)
    return full_dataset


In [25]:
seizure_info

{'chb01_01.edf': [],
 'chb01_02.edf': [],
 'chb01_03.edf': [(2996.0, 3036.0)],
 'chb01_04.edf': [(1467.0, 1494.0)],
 'chb01_05.edf': [],
 'chb01_06.edf': [],
 'chb01_07.edf': [],
 'chb01_08.edf': [],
 'chb01_09.edf': [],
 'chb01_10.edf': [],
 'chb01_11.edf': [],
 'chb01_12.edf': [],
 'chb01_13.edf': [],
 'chb01_14.edf': [],
 'chb01_15.edf': [(1732.0, 1772.0)],
 'chb01_16.edf': [(1015.0, 1066.0)],
 'chb01_17.edf': [],
 'chb01_18.edf': [(1720.0, 1810.0)],
 'chb01_19.edf': [],
 'chb01_20.edf': [],
 'chb01_21.edf': [(327.0, 420.0)],
 'chb01_22.edf': [],
 'chb01_23.edf': [],
 'chb01_24.edf': [],
 'chb01_25.edf': [],
 'chb01_26.edf': [(1862.0, 1963.0)],
 'chb01_27.edf': [],
 'chb01_29.edf': [],
 'chb01_30.edf': [],
 'chb01_31.edf': [],
 'chb01_32.edf': [],
 'chb01_33.edf': [],
 'chb01_34.edf': [],
 'chb01_36.edf': [],
 'chb01_37.edf': [],
 'chb01_38.edf': [],
 'chb01_39.edf': [],
 'chb01_40.edf': [],
 'chb01_41.edf': [],
 'chb01_42.edf': [],
 'chb01_43.edf': [],
 'chb01_46.edf': []}